In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import datetime
import os

import glob
import re

In [2]:
PORT = 8018
LOG_FILE = 'log.csv'
MAX_LOG_SIZE = 100

In [3]:
log_counter = 0
last_rotation_time = None

In [4]:
class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("localhost", PORT),
                            requestHandler=RequestHandler,
                            logRequests=True)

#### Функции ротации журнала

In [5]:
def count_log_lines():
    """Считает количество строк в текущем файле журнала."""
    
    global log_counter
    
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r', encoding='utf-8-sig') as f:
            log_counter = sum(1 for _ in f)
    else:
        log_counter = 0
    return log_counter

In [6]:
def rotate_log_if_needed():
    """Проверяет размер журнала и выполняет ротацию при необходимости."""
    
    global log_counter, last_rotation_time
    
    if log_counter == 0:
        count_log_lines()
    
    if log_counter >= MAX_LOG_SIZE:
        rotation_time = datetime.datetime.now()
        timestamp = rotation_time.strftime("%Y%m%d_%H%M%S")
        archive_name = f"log_{timestamp}.csv"
        
        if os.path.exists(LOG_FILE):
            os.rename(LOG_FILE, archive_name)
            print(f"Журнал ротирован: {LOG_FILE} -> {archive_name}")
        
        # Обновление времени последней ротации
        last_rotation_time = rotation_time
        
        log_counter = 0
        
        return True
    return False

#### Функции работы с журналом

*Дополнительные функции* для получения полной информации при **срезах журнала по времени**

In [7]:
def parse_log_filename(filename):
    """Извлекает дату создания из имени архивного файла лога."""
    # Поиск по шаблону log_YYYYMMDD_hhmmss.csv
    match = re.search(r'log_(\d{8}_\d{6})\.csv', filename)
    if match:
        timestamp_str = match.group(1)
        return datetime.datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")
    return None

In [8]:
def read_log_file(filename):
    """Читает один файл лога и возвращает список кортежей (event_type, timestamp)."""
    entries = []
    if not os.path.exists(filename):
        return entries
    
    with open(filename, 'r', encoding='utf-8-sig') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split(',', 1)
                if len(parts) == 2:
                    entries.append((parts[0], parts[1]))
    return entries

In [9]:
def get_log_files_with_ranges():
    """Находит все файлы логов (текущий + архивные) и определяет их временные диапазоны."""
    # Поиск всех файлов log*.csv
    log_files = glob.glob('log*.csv')
    
    # Разделение на архивные и текущий
    archive_files = []
    current_file = None
    
    for f in log_files:
        basename = os.path.basename(f)
        if basename == LOG_FILE:
            current_file = f
        else:
            dt = parse_log_filename(basename)
            if dt:
                archive_files.append((f, dt))
    
    # Сортировка архивов по времени ротации (от старых к новым)
    archive_files.sort(key=lambda x: x[1])
    
    # Формирование списка с временными диапазонами
    ranges = []
    prev_time = None
    
    for filename, rotation_time in archive_files:
        # Архив содержит записи от prev_time до rotation_time
        ranges.append((filename, prev_time, rotation_time))
        prev_time = rotation_time
    
    # Текущий log.csv — от последней ротации до сейчас
    if current_file:
        ranges.append((current_file, prev_time, None))
    
    return ranges

*Основные функции* работы с **журналом**

In [10]:
def add_log(event_type):
    """
    Добавляет событие в журнал.
    Формат CSV: event_type,YYYY-MM-DD HH:MM:SS
    """
    global log_counter
    
    rotate_log_if_needed()
    
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(LOG_FILE, 'a', encoding='utf-8-sig') as f:
        f.write(f"{event_type},{timestamp}\n")
    
    log_counter += 1
    
    return True

server.register_function(add_log, 'add_log')

<function __main__.add_log(event_type)>

In [11]:
def get_log():
    """Возвращает содержимое только текущего файла log.csv."""
    return read_log_file(LOG_FILE)

server.register_function(get_log, 'get_log')

<function __main__.get_log()>

In [12]:
def get_log_by_type(event_type):
    """Возвращает срез журнала по типу события."""
    all_log = get_log()
    filtered_log = []
    
    for et, ts in all_log:
        # Извлечение базового типа (до двоеточия, если оно есть)
        base_type = et.split(':')[0] if ':' in et else et
        
        if base_type == event_type:
            filtered_log.append((et, ts))
    
    return filtered_log

server.register_function(get_log_by_type, 'get_log_by_type')

<function __main__.get_log_by_type(event_type)>

In [13]:
def get_log_by_time(start_time, end_time):
    """Возвращает срез журнала по времени, читая только нужные файлы."""
    # Обновление временных границ запроса
    try:
        start_dt = datetime.datetime.strptime(start_time, "%Y-%m-%d %H:%M:%S")
        end_dt = datetime.datetime.strptime(end_time, "%Y-%m-%d %H:%M:%S")
    except ValueError as e:
        return f"Ошибка формата времени: {e}"
    
    if start_dt > end_dt:
        return "Ошибка: start_time должно быть раньше end_time"
    
    # Получение временных диапазонов всех файлов
    file_ranges = get_log_files_with_ranges()
    
    filtered_log = []
    
    for filename, file_start, file_end in file_ranges:
        
        if file_end is not None and file_end < start_dt:
            continue
        
        if file_start is not None and file_start > end_dt:
            continue
        
        entries = read_log_file(filename)
        
        for event_type, timestamp_str in entries:
            try:
                log_dt = datetime.datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S")
                if start_dt <= log_dt <= end_dt:
                    filtered_log.append((event_type, timestamp_str))
            except ValueError:
                continue  # Пропуск некорректные записи
    
    # Сортировка результата по времени
    filtered_log.sort(key=lambda x: x[1])
    
    return filtered_log

server.register_function(get_log_by_time, 'get_log_by_time')

<function __main__.get_log_by_time(start_time, end_time)>

In [14]:
def get_log_info():
    """
    Возвращает информацию о всех файлах журнала:
    - список архивных файлов с их временными диапазонами
    - информацию о текущем log.csv
    
    Формат возврата:
    {
        'archives': [(filename, start_time, end_time), ...],
        'current': (filename, start_time, end_time)
    }
    """
    file_ranges = get_log_files_with_ranges()
    
    archives = []
    current = None
    
    for filename, file_start, file_end in file_ranges:
        basename = os.path.basename(filename)
        start_str = file_start.strftime("%Y-%m-%d %H:%M:%S") if file_start else ""
        end_str = file_end.strftime("%Y-%m-%d %H:%M:%S") if file_end else ""
        
        if basename == LOG_FILE:
            current = (basename, start_str, end_str)
        else:
            archives.append((basename, start_str, end_str))
    
    return {'archives': archives, 'current': current}

server.register_function(get_log_info, 'get_log_info')

<function __main__.get_log_info()>

#### Запуск сервера

In [ ]:
count_log_lines()

print(f"Listening on port {PORT}...")
print(f"Max log size: {MAX_LOG_SIZE} entries")
print(f"Current log entries: {log_counter}")

server.serve_forever()

Listening on port 8018...
Max log size: 100 entries
Current log entries: 46


127.0.0.1 - - [04/Oct/2026 00:06:28] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:31] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:35] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:35] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 00:06:38] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/202